[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/project-walkthrough/02_preprocess/02_preprocess_solutions.ipynb)

# 02. `preprocess-example` 동행 — 연습 문제 해설

> 본문: [02_preprocess.ipynb](02_preprocess.ipynb)

먼저 직접 풀어본 뒤에 보세요.

## 0. 환경 준비 — 프로젝트를 옆에 펼쳐두기

In [ ]:
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
print("Colab에서 실행 중:", IN_COLAB)

if IN_COLAB:
    # 이 노트북은 "예제 프로젝트를 옆에 두고 같이 읽는" 노트북입니다.
    # 그래서 설명만 하지 않고, 저장소를 통째로 내려받아 **실제 프로젝트 파일**을 열어봅니다.
    subprocess.run(["git", "clone", "-q", "https://github.com/karzit/temp.git", "/content/temp"], check=False)
    REPO_ROOT = "/content/temp"
    !pip install -q kiwipiepy langchain-text-splitters langchain-core scikit-learn python-dotenv psycopg2-binary
else:
    # 로컬에서 열었다면 이 노트북 위치(notebooks/project-walkthrough/NN_xxx/)에서 3단계 위가 저장소 루트입니다.
    REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", "..", ".."))

PROJECT = os.path.join(REPO_ROOT, "example-projects", "preprocess-example")
SRC = os.path.join(PROJECT, "src")
print("프로젝트 경로:", PROJECT)
assert os.path.isdir(SRC), "프로젝트 경로를 찾지 못했습니다. 저장소 루트에서 노트북을 열었는지 확인하세요."

아래 `show()`는 이 노트북 전체에서 쓰는 도우미입니다. **설명 대신 진짜 프로젝트 파일을 그대로 출력**해서, 노트북과 코드가 어긋나지 않게 합니다.

In [ ]:
import re


def show(filename, start=None, end=None, grep=None):
    """프로젝트 파일의 실제 소스를 줄 번호와 함께 출력한다.

    설명을 읽는 것과 실제 코드를 보는 것 사이의 간격을 없애기 위한 도우미입니다.
    이 노트북에서 "코드 읽기"라고 나오는 곳은 전부 진짜 프로젝트 파일을 그대로 보여줍니다.

        show("crawl.py")                  전체
        show("crawl.py", 30, 45)          30~45번째 줄
        show("crawl.py", grep="def ")     'def '가 들어간 줄만
    """
    path = os.path.join(SRC, filename) if not os.path.isabs(filename) else filename
    lines = open(path, encoding="utf-8").read().splitlines()

    if grep:
        picked = [(i, l) for i, l in enumerate(lines, 1) if re.search(grep, l)]
    else:
        s = (start or 1) - 1
        e = end or len(lines)
        picked = [(i, l) for i, l in enumerate(lines[s:e], s + 1)]

    for i, line in picked:
        print(f"{i:>4} | {line}")


def show_file(relpath, **kwargs):
    """프로젝트 루트 기준 경로로 파일을 보여준다 (README, docker-compose 등)."""
    show(os.path.join(PROJECT, relpath), **kwargs)


# 프로젝트 소스를 import할 수 있도록 경로를 등록해둡니다.
if SRC not in sys.path:
    sys.path.insert(0, SRC)

In [ ]:
os.environ.setdefault("OPENAI_API_KEY", "sk-dummy-not-used")

## 연습 1. 청크에 순번 붙이기

**문제**: 메타데이터에 "이 문서의 3번째 청크"라는 정보를 넣으려면? 그게 있으면 무엇을 할 수 있을까요?

**답부터**: **앞뒤 청크를 같이 보여줄 수 있습니다.**

검색은 청크 단위로 되지만, 답이 청크 경계에 딱 걸치는 경우가 있습니다.
"연차는 15일 주어진다"가 3번 청크 끝에 있고 "단, 3년 이상이면 가산한다"가 4번 청크 시작에 있으면,
3번만 찾아온 AI는 **틀린 답**을 합니다. 순번이 있으면 3번을 찾았을 때 2·4번을 같이 넘길 수 있습니다.
이걸 컨텍스트 확장(context expansion)이라고 합니다.

In [ ]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

text = (
    "제13조(연차유급휴가) 회사는 1년간 80퍼센트 이상 출근한 사원에게 15일의 유급휴가를 준다. "
    "계속하여 근로한 기간이 1년 미만인 사원에게는 1개월 개근 시 1일의 유급휴가를 준다. "
    "3년 이상 계속하여 근로한 사원에게는 매 2년에 대하여 1일을 가산한 유급휴가를 주며, "
    "가산휴가를 포함한 총 휴가일수는 25일을 한도로 한다."
)

splitter = RecursiveCharacterTextSplitter(chunk_size=80, chunk_overlap=0, separators=[". ", " ", ""])
chunks = splitter.split_documents([Document(page_content=text, metadata={"source": "취업규칙.pdf"})])

# 순번 부여: enumerate로 붙이고, 전체 개수도 함께 넣는다.
# total이 있어야 "마지막 청크인가?"를 판단할 수 있어 확장 로직에서 범위를 벗어나지 않는다.
for i, chunk in enumerate(chunks):
    chunk.metadata["chunk_index"] = i
    chunk.metadata["chunk_total"] = len(chunks)

for chunk in chunks:
    meta = chunk.metadata
    print(f"[{meta['chunk_index']}/{meta['chunk_total'] - 1}] {chunk.page_content[:45]}...")

In [ ]:
def expand_context(chunks, hit_index, window=1):
    """검색된 청크의 앞뒤 window개를 함께 돌려준다.

    max/min으로 범위를 자르는 것이 중요하다. 0번 청크를 찾았을 때 -1번을 가져오려 하면
    파이썬 리스트는 조용히 '맨 끝 청크'를 돌려주기 때문에 엉뚱한 내용이 딸려온다.
    """
    start = max(0, hit_index - window)
    end = min(len(chunks), hit_index + window + 1)
    return chunks[start:end]


print("검색 결과가 1번 청크였다고 가정하고 앞뒤를 붙이면:\n")
for chunk in expand_context(chunks, hit_index=1):
    print(f"  [{chunk.metadata['chunk_index']}] {chunk.page_content}")

**결과를 읽는 법**: 1번만 찾았어도 0·1·2번이 같이 나옵니다.
이제 "15일"과 "25일 한도"가 한 컨텍스트 안에 들어와서 AI가 온전한 답을 할 수 있습니다.

> 💡 **트레이드오프**: `window`를 키우면 문맥은 살지만 LLM에게 넘기는 토큰이 늘어납니다.
> 그리고 관련 없는 내용이 섞이면 오히려 답이 흐려집니다.
> 04 노트북에서 다루는 **조항 단위 청킹**은 이 문제를 다른 방식으로 풉니다 —
> 애초에 의미 경계로 자르면 확장이 덜 필요해집니다.

## 연습 2. 정제 규칙 추가하기 — 그리고 과하게 지우면 생기는 일

**문제**: `[목차로]`, `인쇄하기` 같은 UI 텍스트를 걸러내세요. 과하게 지우면 어떤 문제가 생길까요?

In [ ]:
import re

from preprocess import clean_text

UI_NOISE_PATTERNS = [
    r"^\s*\[?(목차로|맨\s*위로|이전|다음)\]?\s*$",  # 한 줄 전체가 네비게이션인 경우만
    r"^\s*(인쇄하기|공유하기|스크랩)\s*$",
    r"^\s*조회수\s*[:：]?\s*[\d,]+\s*$",
]


def clean_text_v2(text: str) -> str:
    """기존 정제에 UI 텍스트 제거를 얹는다.

    핵심은 **줄 단위로, 그 줄 전체가 UI 문구일 때만** 지우는 것이다.
    (^...$ 와 re.M을 쓴 이유가 이것이다.)
    """
    text = clean_text(text)
    for pattern in UI_NOISE_PATTERNS:
        text = re.sub(pattern, "", text, flags=re.M)
    return re.sub(r"\n{3,}", "\n\n", text).strip()


sample = """[목차로]
제15조(병가)
① 사원이 업무 외의 질병으로 근무할 수 없는 경우 병가를 부여할 수 있다.
조회수: 1,024
인쇄하기
② 병가 기간 중의 임금은 지급하지 아니한다."""

print("=== 정제 후 ===")
print(clean_text_v2(sample))

### 과하게 지우면 생기는 일

`^...$`를 빼고 그냥 단어만 지우면 어떻게 되는지 직접 보세요.

In [ ]:
danger = "제20조(인쇄하기) 회사는 문서의 인쇄하기 기능을 다음 각 호에 따라 제한한다."

print("줄 전체 매칭(안전):")
print("  ", re.sub(r"^\s*인쇄하기\s*$", "", danger, flags=re.M))

print("\n단어 매칭(위험):")
print("  ", re.sub(r"인쇄하기", "", danger))

**규정 조문 자체가 사라졌습니다.** `제20조(인쇄하기)`는 진짜 조항인데 제목이 통째로 날아갔습니다.

이게 정제의 딜레마입니다.

| | 덜 지우면 | 더 지우면 |
|---|---|---|
| 결과 | 잡음이 검색에 걸림 | **원문이 손상됨** |
| 복구 | 나중에 규칙을 추가하면 됨 | **원본이 남아 있어야만 복구 가능** |

**그래서 01번 프로젝트가 원본을 그대로 보관하는 겁니다.** 정제는 언제든 다시 할 수 있어야 하고,
그러려면 손대지 않은 원본이 어딘가에 있어야 합니다. 세 프로젝트가 이렇게 이어집니다.

원칙: **애매하면 덜 지웁니다.** 잡음은 검색 점수를 조금 떨어뜨리지만, 손상된 원문은 답을 틀리게 만듭니다.

## 연습 3. 키워드를 검색에 실제로 써보기

**문제**: `extract_keywords()`가 뽑은 키워드로 검색을 만들고 TF-IDF와 비교하세요.

In [ ]:
from preprocess import extract_keywords

docs = [
    "제9조(근로시간) 1주간의 소정근로시간은 휴게시간을 제외하고 40시간으로 한다.",
    "제10조(휴게) 회사는 근로시간이 4시간인 경우에는 30분 이상의 휴게시간을 부여한다.",
    "제13조(연차유급휴가) 1년간 80퍼센트 이상 출근한 사원에게 15일의 유급휴가를 준다.",
    "제14조(경조휴가) 본인의 결혼은 5일, 배우자의 출산은 10일의 유급휴가를 부여한다.",
]

# 문서마다 명사를 미리 뽑아둡니다 (색인 시점에 하는 일).
doc_keywords = [set(extract_keywords(d, limit=20)) for d in docs]
for d, kws in zip(docs, doc_keywords):
    print(f"{d[:22]}... -> {sorted(kws)}")

In [ ]:
def keyword_search(question, k=2):
    """질문에서도 명사를 뽑아, 문서 키워드와 겹치는 개수로 점수를 매긴다."""
    q_keywords = set(extract_keywords(question, limit=20))
    scored = []
    for doc, kws in zip(docs, doc_keywords):
        overlap = q_keywords & kws
        scored.append((len(overlap), overlap, doc))
    scored.sort(key=lambda x: x[0], reverse=True)
    return scored[:k]


for question in ["쉬는 시간은 얼마나 주나요?", "결혼하면 며칠 쉬어요?"]:
    print(f"질문: {question}")
    print(f"  질문 명사: {sorted(set(extract_keywords(question, limit=20)))}")
    for score, overlap, doc in keyword_search(question):
        print(f"  [{score}점] {doc[:34]}...  겹친 명사={sorted(overlap)}")
    print()

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

vectorizer = TfidfVectorizer()
matrix = vectorizer.fit_transform(docs)

for question in ["쉬는 시간은 얼마나 주나요?", "결혼하면 며칠 쉬어요?"]:
    scores = cosine_similarity(vectorizer.transform([question]), matrix)[0]
    top = scores.argsort()[::-1][:2]
    print(f"질문: {question}")
    for i in top:
        print(f"  [{scores[i]:.3f}] {docs[i][:34]}...")
    print()

**결과를 읽는 법 — 두 방식을 비교하세요.**

"쉬는 시간"은 문서에 **"휴게시간"**으로 적혀 있습니다. 글자가 하나도 안 겹칩니다.

- **TF-IDF**: 단어가 겹쳐야 점수가 나오므로 0점에 가깝습니다. 못 찾습니다.
- **명사 매칭**: "시간"이 겹쳐서 점수가 조금 나옵니다. TF-IDF보다는 낫지만 여전히 약합니다.

**둘 다 "쉬는 = 휴게"라는 걸 모릅니다.** 글자를 보기 때문입니다.
이 문제를 푸는 게 **임베딩**이고, 04 노트북에서 다시 다룹니다.

그럼 명사 추출은 왜 하나요? **조사 문제를 해결하기 때문입니다.**
"휴가를"/"휴가는"/"휴가가"를 하나로 묶어주는 건 임베딩이 아니라 형태소 분석의 몫입니다.
실제 시스템은 **둘을 같이 씁니다** — 형태소 분석으로 키워드 검색을 정확하게 만들고,
임베딩으로 의미 검색을 더하고, 04에서 배울 RRF로 합칩니다.

## 정리

| 연습 | 핵심 |
|---|---|
| 청크 순번 | 검색은 청크 단위, 답은 경계에 걸칠 수 있음 → 앞뒤 확장 |
| 정제 규칙 | **애매하면 덜 지운다.** 손상된 원문은 복구 불가 |
| 키워드 검색 | 형태소 분석은 조사 문제를, 임베딩은 동의어 문제를 푼다. 서로 다른 문제 |

본문으로 돌아가기: [02_preprocess.ipynb](02_preprocess.ipynb)